# Imports

In [1]:
import numpy as np
from scipy.spatial.transform import Rotation as rotate

from functions.representation import Object, RectangularPrism, find_center_point_LWLC, load_objects
from functions.vectors import cosine_similarity, find_axis_of_rotation_geon_only, find_axis_of_rotation_geon_and_spatcon, same_object
from functions.plotting import  add_point, add_landmarks, add_axis, add_object, add_frame, prepare_rotation_graphs

import plotly.graph_objects as go
import plotly.express as px
import plotly.offline as pyo
from plotly.subplots import make_subplots
import math, copy

# Initialize Plotly for offline mode in Jupyter Notebook
pyo.init_notebook_mode(connected=True)

# Model Variables

In [2]:
total_run_time = 0

# timing
production_time = 50                        # 50ms
propositional_difficulty_time = 3           # 1-3 ms
object_encoding_time = 300                  # 150-300ms for each object

# rotation
max_step_size = 30
min_step_size = 1
speed_constant = 3 # idek like 2-4?

# decision
difference_confidence = 0
difference_confidence_threshold = 2         # 2-4 repeated steps

# similarity thresholds
geon_alignment_threshold = 0.8  # maybe 0.7 - 0.9
landmark_angle_threshold = 0.99
landmark_distance_threshold = 0.4               # lowkey this has to be higher as speed constant gets higher --> unless switch is implmented UGH
object_angle_threshold = 0.975
object_distance_threshold = landmark_distance_threshold * 2

# Model Functions

In [3]:
# rotation
# def calculate_step_size_v1(angular_disparity):
#     step_size = angular_disparity * step_size_decrease
#     step_size = min(max(step_size, min_step_size), max_step_size)

#     return step_size

# tbh i like this the best - others get too slow too quick
def calculate_step_size_simple(angular_disparity):
    # step_size = speed_constant * math.sqrt(angular_disparity)
    step_size = speed_constant * math.sqrt(angular_disparity)


    return step_size

def calculate_step_size_min_jerk_trajectory(angular_disparity, total_angular_disparity):
    x = angular_disparity/total_angular_disparity
    print(total_angular_disparity)
    step_size = 20*((30 * x**2) - (60 * x**3) + (30 * x**4))

    return min(max(step_size, min_step_size), max_step_size)

# def calculate_step_size_fitts_law(angular_disparity):
def fitts_step(theta, dt, a=0.1, b=0.2, epsilon=0.01):
    """
    theta: remaining angular disparity, in radians
    dt: timestep
    a, b: fitted constants
    epsilon: angular tolerance, in radians
    """

    theta = abs(theta)

    if theta <= epsilon:
        return theta

    ID = np.log2(theta / epsilon + 1)
    MT = a + b * ID

    omega = theta / MT
    step = omega * dt

    return min(step, theta)


# 1. Representation

This is the case I'll model:

![image](test.jpg)

*150 degree diff in pic

In [4]:
# r = rotate.from_euler('z', 60, degrees=True)                                  # 60 deg rotation around z-axis
# r = rotate.from_rotvec([0, 0, np.deg2rad(150)])                               # 150 deg rotation around z-axis
# r = rotate.from_rotvec([0, 0, np.deg2rad(-219)])                              # -219 deg rotation around z-axis
# r = rotate.from_rotvec([0, np.deg2rad(190), 0])                               # 180 deg rotation around y-axis
# r = rotate.from_rotvec([0, np.deg2rad(220), 0])                               # 220 deg rotation around y-axis
# r = rotate.from_rotvec([0, np.deg2rad(-60), 0])                               # -60 deg rotation around y-axis
# r = rotate.from_rotvec([np.deg2rad(150), 0, 0])                               # 150 deg rotation around x-axis
# r = rotate.from_rotvec([0, np.deg2rad(60), np.deg2rad(30)])                   # 60 deg rotation around y-axis, 30 deg rotation around z-axis
# r = rotate.from_rotvec([0, np.deg2rad(180), np.deg2rad(180)])                 # 180 deg rotation around y-axis, 180 deg rotation around z-axis
r = rotate.from_rotvec([np.deg2rad(150), np.deg2rad(60), np.deg2rad(90)])       # crazy rotation :o
# r = rotate.from_rotvec([np.deg2rad(150), np.deg2rad(100), np.deg2rad(130)])       # crazy rotation :o
# r = rotate.from_rotvec([np.deg2rad(90), np.deg2rad(100), np.deg2rad(90)])       # crazy rotation :o

# x: right is positive, y: further away is positive, z: up is positive

# # create original geons
# g1 = RectangularPrism(2, np.array([-1, -1, 0]))             # 45 deg angle front left, no z info
# g2 = RectangularPrism(3, np.array([0, 0, -1]))              # down
# # g1 = RectangularPrism(2, np.array([0.02, 0.0, 1.0]))      # 45 deg angle front left, no z info
# # g2 = RectangularPrism(3, np.array([-0.02, 0.0, 2.0]))     # down
# g3 = RectangularPrism(2, np.array([1, 1, 0]))               # 45 deg angle back right, no z info
# g4 = RectangularPrism(1, np.array([1, -1, 0]))              # 45 deg angle front right, no z info

# # create original object and relations
# original_object = Object(
# # target_object = Object(
#     geons = [g1,g2,g3,g4],
#     landmark_geon_index = 0                    # i.e. landmark is g1
# )

# # create target geons (same as original, but with rotation applied)
# g1 = RectangularPrism(2, r.apply(np.array([-1, -1, 0])))
# g2 = RectangularPrism(3, r.apply(np.array([0, 0, -1])))
# # g1 = RectangularPrism(2, r.apply(np.array([0.02, 0.0, 1.0])))
# # g2 = RectangularPrism(3, r.apply(np.array([-0.02, 0.0, 2.0])))
# g3 = RectangularPrism(2, r.apply(np.array([1, 1, 0])))
# g4 = RectangularPrism(1, r.apply(np.array([1, -1, 0])))

# # # MIRRORED target object
# # g1 = RectangularPrism(2, r.apply(np.array([-1, -1, 0])))
# # g2 = RectangularPrism(3, r.apply(np.array([0, 0, -1])))
# # g3 = RectangularPrism(2, r.apply(np.array([1, 1, 0])))
# # g4 = RectangularPrism(1, r.apply(np.array([-1, 1, 0])))

# target_object = Object(
# # original_object = Object(
#     geons = [g1,g2,g3,g4],
#     landmark_geon_index = 0                    # i.e. landmark is g1
# )

# LOADED OBJECTS

objects = load_objects("JostJansenShapes.txt")
original_object = copy.deepcopy(objects[1])
target_object = copy.deepcopy(objects[1])
target_object.rotate(r)

In [5]:
# add time needed for representation
total_run_time = total_run_time + (2 * object_encoding_time)

In [6]:
# Create graph
fig = make_subplots(rows=1, cols=2, specs=[[{'type': 'scene'}, {'type': 'scene'}]], subplot_titles=("Original Object", "Target Object"))

add_object(fig, original_object, "Original Object", colour='blue', row=1, col=1)
add_object(fig, target_object, "Target Object", colour='red', row=1, col=2)

fig.update_layout(title='3D Vector Visualization', showlegend=False)

# 2. Landmarking

In [7]:
# calculate axis of rotation, direction of rotation, and total angular disparity
center_point = np.array([0,0,0])
axis_of_rotation, direction, angle = find_axis_of_rotation_geon_only(original_object, target_object, center_coords=center_point)
total_angular_disparity = angle
print(axis_of_rotation)

# add time needed for landmarking
total_run_time = total_run_time + production_time                                                                 # check geon
total_run_time = total_run_time + production_time + (total_angular_disparity * propositional_difficulty_time)     # check spatial connection

[ 0.24791987  0.         -0.57285684]


In [8]:
# Create graph
axis_fig = go.Figure(data=[])

print(original_object.get_landmark_endpoints())

add_object(axis_fig, original_object, "Original Object", colour='blue')
add_object(axis_fig, target_object, "Target Object", colour='red')

add_landmarks(axis_fig, original_object.get_landmark_endpoints(), "Original Landmarks", colour='purple')
add_landmarks(axis_fig, target_object.get_landmark_endpoints(), "Target Landmarks", colour='orange')
add_point(axis_fig, center_point, "Rotation Point", colour='green')

add_axis(axis_fig, axis_of_rotation, scale=2)

axis_fig.update_layout(title='3D Vector Visualization')

[[ 0. -2.  0.]
 [ 0.  0.  0.]
 [ 0.  0. -1.]]


# 3. Rotation

In [9]:
# prepare graphs
axis_animation_fig, overlap_animation_fig, sidebyside_animation_fig = prepare_rotation_graphs(original_object, target_object, axis_of_rotation, center_point, production_time)
axis_animation = []
overlap_animation = []
sidebyside_animation = []

# set landmark vectors, angular disparity
original_landmark_geon_vector = original_object.get_landmark_geon().get_vector()
target_landmark_geon_vector = target_object.get_landmark_geon().get_vector()
original_spatcon_direction = original_object.get_landmark_spatial_connection().get_vector()
target_spatcon_direction = target_object.get_landmark_spatial_connection().get_vector()
curr_step_angular_disparity = total_angular_disparity
total_angular_disparity_part2 = None
switch_direction = False

# STEP ONE: GEON ALIGNMENT
print("STEP ONE: GEON ALIGNMENT")

loop_count = 0
while cosine_similarity(original_landmark_geon_vector, target_landmark_geon_vector) < geon_alignment_threshold:     # checking cosine similarity between target and goal geons

    # find best axis/direction of rotation, angular disparity
    axis_of_rotation, direction, curr_step_angular_disparity = find_axis_of_rotation_geon_only(original_object, target_object, center_coords=center_point, prev_axis=axis_of_rotation, prev_direction=direction, prev_angle=curr_step_angular_disparity, total_angular_disparity=total_angular_disparity)

    # calculate step size based on angular disparity
    # step_size = calculate_step_size_min_jerk_trajectory(curr_step_angular_disparity, total_angular_disparity)
    # step_size = fitts_step(curr_step_angular_disparity, loop_count)
    step_size = calculate_step_size_simple(curr_step_angular_disparity)
    print("CURR_ANGULAR_DISPARITY: " + str(curr_step_angular_disparity) + ", STEP_SIZE: " + str(step_size))

    # apply rotation to original object
    r = rotate.from_rotvec(direction * np.deg2rad(step_size) * axis_of_rotation)        # quaternion representing step size rotation around calculated axis
    original_object.rotate(r)

    # update original geon and spatial connection vectors
    original_landmark_geon_vector = original_object.get_landmark_geon().get_vector()
    original_spatcon_direction = original_object.get_landmark_spatial_connection().get_vector()

    # add animation frames to graphs
    add_frame(axis_animation, original_object, object_name="Original Object", landmark_name="Original Landmarks", axis_of_rotation=axis_of_rotation, object_colour='blue', landmark_colour='purple', axis_colour='green', axis_scale=2)
    original_centerpoint_vec = find_center_point_LWLC(original_object)
    copy_og_obj = copy.deepcopy(original_object)
    copy_og_obj.update_start_coords(copy_og_obj.start_coords - original_centerpoint_vec)
    add_frame(overlap_animation, copy_og_obj, object_name="Original Object", object_colour='blue', axis_scale=2)
    add_frame(sidebyside_animation, copy_og_obj, object_name="Original Object", object_colour='blue', axis_scale=2)

    loop_count += 1

    # emergency break
    if loop_count > 100:
        break

# STEP TWO: GEON AND SPAT CON ALIGNMENT:
print("\nSTEP TWO: GEON AND SPAT CON ALIGNMENT")

loop_count = 0
while cosine_similarity(original_landmark_geon_vector, target_landmark_geon_vector) < landmark_angle_threshold or cosine_similarity(original_spatcon_direction, target_spatcon_direction) < landmark_angle_threshold:     # checking cosine similarity between target and goal geons and spatial connections

    # find best axis/direction of rotation, angular disparity
    axis_of_rotation, direction, curr_step_angular_disparity = find_axis_of_rotation_geon_and_spatcon(original_object, target_object, center_coords=center_point, prev_axis=axis_of_rotation, prev_direction=direction, prev_angle=curr_step_angular_disparity, total_angular_disparity=total_angular_disparity)

    # calculate step size based on angular disparity
    if total_angular_disparity_part2 == None:
        total_angular_disparity_part2 = curr_step_angular_disparity
    # step_size = calculate_step_size_min_jerk_trajectory(curr_step_angular_disparity, total_angular_disparity_part2)
    # step_size = fitts_step(curr_step_angular_disparity, loop_count)
    step_size = calculate_step_size_simple(curr_step_angular_disparity)
    print("CURR_ANGULAR_DISPARITY: " + str(curr_step_angular_disparity) + ", STEP_SIZE: " + str(step_size))

    # apply rotation to original object
    r = rotate.from_rotvec(direction * np.deg2rad(step_size) * axis_of_rotation)        # quaternion representing step size rotation around calculated axis
    original_object.rotate(r)

    # update original geon and spatial connection vectors
    original_landmark_geon_vector = original_object.get_landmark_geon().get_vector()
    original_spatcon_direction = original_object.get_landmark_spatial_connection().get_vector()

    # add animation frames to graphs
    add_frame(axis_animation, original_object, object_name="Original Object", landmark_name="Original Landmarks", axis_of_rotation=axis_of_rotation, object_colour='blue', landmark_colour='purple', axis_colour='green', axis_scale=2)
    original_centerpoint_vec = find_center_point_LWLC(original_object)
    copy_og_obj = copy.deepcopy(original_object)
    copy_og_obj.update_start_coords(copy_og_obj.start_coords - original_centerpoint_vec)
    add_frame(overlap_animation, copy_og_obj, object_name="Original Object", object_colour='blue', axis_scale=2)
    add_frame(sidebyside_animation, copy_og_obj, object_name="Original Object", object_colour='blue', axis_scale=2)

    loop_count += 1

    # emergency break
    if loop_count > 100:
        break

# find_axis_of_rotation_geon_and_spatcon

#  or cosine_similarity(original_spatcon_direction, target_spatcon_direction) < landmark_angle_threshold

axis_animation_fig.frames = axis_animation
axis_animation_fig.show()
overlap_animation_fig.frames = overlap_animation
overlap_animation_fig.show()
sidebyside_animation_fig.frames = sidebyside_animation
sidebyside_animation_fig.show()

STEP ONE: GEON ALIGNMENT
CURR_ANGULAR_DISPARITY: 128.17302352858704, STEP_SIZE: 33.96405764565364
CURR_ANGULAR_DISPARITY: 107.05390196036906, STEP_SIZE: 31.040056663017253
CURR_ANGULAR_DISPARITY: 77.15922431689127, STEP_SIZE: 26.35209704847076
CURR_ANGULAR_DISPARITY: 57.52502205820524, STEP_SIZE: 22.75357551075978

STEP TWO: GEON AND SPAT CON ALIGNMENT
CURR_ANGULAR_DISPARITY: 165.319043181563, STEP_SIZE: 38.572935960775226
CURR_ANGULAR_DISPARITY: 138.3046748772505, STEP_SIZE: 35.280902396271756
CURR_ANGULAR_DISPARITY: 106.50759305682595, STEP_SIZE: 30.960754795570367
CURR_ANGULAR_DISPARITY: 76.73370228765806, STEP_SIZE: 26.2793325750279
CURR_ANGULAR_DISPARITY: 50.93223375969742, STEP_SIZE: 21.410046796709175
CURR_ANGULAR_DISPARITY: 29.760478029861307, STEP_SIZE: 16.36594947654281
CURR_ANGULAR_DISPARITY: 13.562544739466981, STEP_SIZE: 11.048208119654646


# 4. Decision

In [10]:
# check overall similarity

are_same, run_time = same_object(original_object, target_object, object_angle_threshold, total_angular_disparity, production_time, propositional_difficulty_time)
total_run_time += run_time

if are_same:
    print("SAME")
else:
    print("DIFFERENT")
    # repeat rotation
    # repeat landmarking if needed --> skip for now

    # mayve do: how many repetitions before confidence needed? low confidence needed person: 0, high confidence needed person: 2-3

SAME
